In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

# 1. Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. Transform: normalize to [0,1], flatten later
transform = transforms.Compose([
    transforms.ToTensor(),  # Converts to [0,1]
    transforms.Lambda(lambda x: x.view(-1))  # Flatten 28x28 -> 784
])

# 3. Load MNIST dataset
train_dataset = datasets.MNIST(root="./data", train=True, transform=transform, download=False)
test_dataset = datasets.MNIST(root="./data", train=False, transform=transform, download=False)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)



In [ ]:
import numpy as np

# ---------------- Load Dataset ----------------
# Assume X_train, y_train, X_test, y_test are already loaded
# X: shape (N, 28, 28), y: shape (N,)
# Normalize

def one_hot(y, num_classes=10):
    oh = np.zeros((y.size, num_classes))
    oh[np.arange(y.size), y] = 1
    return oh

# ---------------- Layers ----------------
class Conv2D:
    def __init__(self, num_filters, filter_size, input_channels):
        self.num_filters = num_filters
        self.filter_size = filter_size
        self.input_channels = input_channels
        scale = 1.0 / np.sqrt(filter_size * filter_size * input_channels)
        self.filters = np.random.randn(num_filters, input_channels, filter_size, filter_size) * scale
        self.bias = np.zeros((num_filters, 1))
    
    def forward(self, x):
        self.x = x
        N, C, H, W = x.shape
        F, _, HH, WW = self.filters.shape
        out_h = H - HH + 1
        out_w = W - WW + 1
        out = np.zeros((N, F, out_h, out_w))
        
        for n in range(N):
            for f in range(F):
                for i in range(out_h):
                    for j in range(out_w):
                        region = x[n, :, i:i+HH, j:j+WW]
                        out[n, f, i, j] = np.sum(region * self.filters[f]) + self.bias[f]
        self.out = out
        return out
    
    def backward(self, d_out, lr=0.01):
        N, F, H, W = d_out.shape
        _, C, HH, WW = self.filters.shape
        dx = np.zeros_like(self.x)
        dW = np.zeros_like(self.filters)
        dB = np.zeros_like(self.bias)
        
        for n in range(N):
            for f in range(F):
                for i in range(H):
                    for j in range(W):
                        region = self.x[n, :, i:i+HH, j:j+WW]
                        dW[f] += d_out[n, f, i, j] * region
                        dx[n, :, i:i+HH, j:j+WW] += d_out[n, f, i, j] * self.filters[f]
                dB[f] += np.sum(d_out[n, f])
        
        self.filters -= lr * dW
        self.bias -= lr * dB
        return dx

class ReLU:
    def forward(self, x):
        self.x = x
        return np.maximum(0, x)
    
    def backward(self, d_out):
        return d_out * (self.x > 0)

class MaxPool2x2:
    def forward(self, x):
        self.x = x
        N, C, H, W = x.shape
        out = np.zeros((N, C, H//2, W//2))
        self.argmax = np.zeros_like(x, dtype=bool)
        
        for n in range(N):
            for c in range(C):
                for i in range(0, H, 2):
                    for j in range(0, W, 2):
                        region = x[n, c, i:i+2, j:j+2]
                        idx = np.unravel_index(np.argmax(region), region.shape)
                        out[n, c, i//2, j//2] = region[idx]
                        self.argmax[n, c, i+idx[0], j+idx[1]] = True
        return out
    
    def backward(self, d_out):
        dx = np.zeros_like(self.x)
        N, C, H2, W2 = d_out.shape
        for n in range(N):
            for c in range(C):
                for i in range(H2):
                    for j in range(W2):
                        dx[n, c, i*2:i*2+2, j*2:j*2+2][self.argmax[n, c, i*2:i*2+2, j*2:j*2+2]] = d_out[n, c, i, j]
        return dx

class Flatten:
    def forward(self, x):
        self.x_shape = x.shape
        return x.reshape(x.shape[0], -1)
    
    def backward(self, d_out):
        return d_out.reshape(self.x_shape)

class Dense:
    def __init__(self, in_dim, out_dim):
        scale = 1.0 / np.sqrt(in_dim)
        self.W = np.random.randn(in_dim, out_dim) * scale
        self.b = np.zeros((1, out_dim))
    
    def forward(self, x):
        self.x = x
        return x @ self.W + self.b
    
    def backward(self, d_out, lr=0.01):
        dW = self.x.T @ d_out
        db = np.sum(d_out, axis=0, keepdims=True)
        dx = d_out @ self.W.T
        self.W -= lr * dW
        self.b -= lr * db
        return dx

class SoftmaxCrossEntropy:
    def forward(self, logits, y_true):
        exps = np.exp(logits - np.max(logits, axis=1, keepdims=True))
        self.probs = exps / np.sum(exps, axis=1, keepdims=True)
        self.y_true = y_true
        loss = -np.mean(np.sum(y_true * np.log(self.probs + 1e-9), axis=1))
        return loss
    
    def backward(self):
        return (self.probs - self.y_true) / self.y_true.shape[0]

# ---------------- Build CNN ----------------
conv = Conv2D(8, 3, 1)
relu = ReLU()
pool = MaxPool2x2()
flat = Flatten()
fc = Dense(13*13*8, 10)
loss_fn = SoftmaxCrossEntropy()

# ---------------- Training ----------------
epo

C:\Users\sshak\AppData\Local\Temp\ipykernel_11396\1158051841.py:36: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  out[n, f, i, j] = np.sum(region * self.filters[f]) + self.bias[f]


Epoch 1, Batch 10/7500, Loss=2.2263
Epoch 1, Batch 20/7500, Loss=2.1238
Epoch 1, Batch 30/7500, Loss=2.0612
Epoch 1, Batch 40/7500, Loss=1.5500
